# 02 - Fine-tune IndoBERTweet (S3) dengan 5-fold CV + Class Weight

Melatih **IndoBERTweet** untuk dua skema label sekaligus — **binary** (negatif vs non-negatif) dan **3class** (negatif/netral/positif) — pakai **5-fold cross-validation** supaya metrik untuk kelas negatif yang sangat sedikit tidak goyang kalau cuma dites di satu hold-out set.

Setiap fold:
- Train di 4 fold (dari situ disisihkan 10% buat validasi/early stopping)
- Evaluasi di 1 fold yang disisihkan (benar-benar belum pernah dilihat model)

Loss function dibobot pakai `class_weights` yang sudah dihitung di notebook sebelumnya (`data/siap_latih/<scheme>/config.json`), supaya model tidak mengabaikan kelas negatif.

Butuh `data/siap_latih/binary/dataset_final.csv` dan `data/siap_latih/3class/dataset_final.csv` dari notebook `01_persiapan_data.ipynb`.

Install dulu kalau belum ada (sesuaikan CUDA versi kamu, cek di https://pytorch.org):
```
pip install torch --index-url https://download.pytorch.org/whl/cu121
pip install transformers scikit-learn pandas
```

In [1]:
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)

# ============================ KONFIGURASI ============================
SCHEMES          = ['binary', '3class']   # dilatih berurutan, satu-satu (hemat VRAM)
N_FOLDS          = 5
EPOCHS           = 4
LR               = 2e-5
BATCH_TRAIN      = 16     # turunkan ke 8 kalau CUDA out of memory di GPU 6GB
BATCH_EVAL       = 32
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.1
EARLY_STOP_PATIENCE = 2
SAVE_MODELS      = False  # True -> simpan model terbaik tiap fold ke models/<scheme>/fold_<i>/ (~440MB x 5 per skema)
SEED             = 42
# =====================================================================

ROOT_DIR   = Path.cwd().parent
DATA_DIR   = ROOT_DIR / 'data' / 'siap_latih'
MODELS_DIR = ROOT_DIR / 'models'
RESULTS_DIR = ROOT_DIR / 'data' / 'hasil_cv'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))
    print('VRAM  : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print('PERHATIAN: tidak terdeteksi GPU. Training akan sangat lambat di CPU.')

c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
PERHATIAN: tidak terdeteksi GPU. Training akan sangat lambat di CPU.


## Cek dataset tersedia untuk kedua skema

In [2]:
configs = {}
for scheme in SCHEMES:
    scheme_dir = DATA_DIR / scheme
    ds_path = scheme_dir / 'dataset_final.csv'
    cfg_path = scheme_dir / 'config.json'
    assert ds_path.exists(), f'Tidak ditemukan: {ds_path} -> jalankan 01_persiapan_data.ipynb dulu'
    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
    configs[scheme] = cfg
    df = pd.read_csv(ds_path)
    print(f'[{scheme}] baris={len(df)}  label2id={cfg["label2id"]}  max_len={cfg["max_len"]}')
    print(f'          class_weights={cfg["class_weights"]}')

[binary] baris=2000  label2id={'non-negatif': 0, 'negatif': 1}  max_len=56
          class_weights={'0': 0.5125628140703518, '1': 20.4}
[3class] baris=2000  label2id={'negatif': 0, 'netral': 1, 'positif': 2}  max_len=56
          class_weights={'0': 13.6, '1': 0.37362637362637363, '2': 4.0}


## Dataset & model utilities

- `CommentDataset`: tokenisasi sekali di awal (bukan on-the-fly), supaya training tiap epoch lebih cepat
- `WeightedTrainer`: subclass `Trainer` yang membobot loss pakai `class_weights`
- `make_compute_metrics`: macro-F1 + F1 per kelas (termasuk F1-negatif, metrik utama untuk kelas minoritas)

In [3]:
class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        enc = tokenizer(
            list(texts), truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt',
        )
        self.input_ids = enc['input_ids']
        self.attention_mask = enc['attention_mask']
        self.labels = torch.tensor(np.asarray(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx],
        }


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def make_compute_metrics(id2label):
    label_names = [id2label[str(i)] if str(i) in id2label else id2label[i] for i in sorted(map(int, id2label))]
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        out = {
            'accuracy': accuracy_score(labels, preds),
            'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        }
        per_class = f1_score(labels, preds, average=None, zero_division=0,
                              labels=list(range(len(label_names))))
        for name, f1_val in zip(label_names, per_class):
            out[f'f1_{name}'] = f1_val
        return out
    return compute_metrics

## Loop 5-fold CV untuk kedua skema

Untuk tiap fold: 4 fold jadi `remain` (dipecah lagi 90/10 jadi train/val internal buat early stopping), 1 fold jadi test (benar-benar held-out).

In [4]:
all_fold_results = []     # ringkasan metrik per fold, untuk kedua skema
all_fold_reports  = {}     # classification_report lengkap per (scheme, fold)

for scheme in SCHEMES:
    cfg = configs[scheme]
    label2id = cfg['label2id']
    id2label = cfg['id2label']
    max_len  = cfg['max_len']
    model_name = cfg['model_name']
    n_labels = len(label2id)
    class_weight_vec = torch.tensor(
        [cfg['class_weights'][str(i)] for i in range(n_labels)], dtype=torch.float
    )

    df_full = pd.read_csv(DATA_DIR / scheme / 'dataset_final.csv')
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    print(f'\n{"="*70}\n  SKEMA: {scheme}  ({n_labels} kelas)\n{"="*70}')

    for fold in range(N_FOLDS):
        t0 = time.time()
        test_df   = df_full[df_full['fold'] == fold].reset_index(drop=True)
        remain_df = df_full[df_full['fold'] != fold].reset_index(drop=True)

        train_df, val_df = train_test_split(
            remain_df, test_size=0.1, random_state=SEED + fold,
            stratify=remain_df['label'],
        )

        train_ds = CommentDataset(train_df['text_model'], train_df['label'], tokenizer, max_len)
        val_ds   = CommentDataset(val_df['text_model'],   val_df['label'],   tokenizer, max_len)
        test_ds  = CommentDataset(test_df['text_model'],  test_df['label'],  tokenizer, max_len)

        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=n_labels)

        out_dir = MODELS_DIR / scheme / f'fold_{fold}'
        total_steps = (len(train_df) // BATCH_TRAIN + 1) * EPOCHS
        warmup_steps = int(total_steps * WARMUP_RATIO)
        args = TrainingArguments(
            output_dir=str(out_dir / 'checkpoints'),
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_TRAIN,
            per_device_eval_batch_size=BATCH_EVAL,
            learning_rate=LR,
            weight_decay=WEIGHT_DECAY,
            warmup_steps=warmup_steps,
            eval_strategy='epoch',
            save_strategy='epoch',
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model='f1_macro',
            greater_is_better=True,
            fp16=(device == 'cuda'),
            logging_steps=50,
            seed=SEED + fold,
            report_to='none',
            disable_tqdm=False,
        )

        trainer = WeightedTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            compute_metrics=make_compute_metrics(id2label),
            class_weights=class_weight_vec,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
        )

        print(f'\n--- {scheme} | fold {fold} | train={len(train_df)} val={len(val_df)} test={len(test_df)} ---')
        trainer.train()

        # evaluasi di test fold yang benar-benar held-out
        pred_out = trainer.predict(test_ds)
        preds = np.argmax(pred_out.predictions, axis=-1)
        labels_true = test_df['label'].to_numpy()

        label_names = [id2label[str(i)] for i in range(n_labels)]
        report = classification_report(labels_true, preds, target_names=label_names,
                                        output_dict=True, zero_division=0)
        cm = confusion_matrix(labels_true, preds, labels=list(range(n_labels)))

        row = {
            'scheme': scheme, 'fold': fold,
            'n_train': len(train_df), 'n_val': len(val_df), 'n_test': len(test_df),
            'accuracy': report['accuracy'],
            'f1_macro': report['macro avg']['f1-score'],
        }
        for name in label_names:
            row[f'f1_{name}']        = report[name]['f1-score']
            row[f'precision_{name}'] = report[name]['precision']
            row[f'recall_{name}']    = report[name]['recall']
        all_fold_results.append(row)
        all_fold_reports[(scheme, fold)] = {'report': report, 'confusion_matrix': cm.tolist(), 'label_names': label_names}

        print(f'  f1_macro={row["f1_macro"]:.3f} | ' + ' | '.join(f'f1_{n}={row[f"f1_{n}"]:.3f}' for n in label_names))
        print(f'  waktu fold: {(time.time()-t0)/60:.1f} menit')

        if SAVE_MODELS:
            out_dir.mkdir(parents=True, exist_ok=True)
            trainer.save_model(str(out_dir))
            tokenizer.save_pretrained(str(out_dir))

        # bersihkan memori sebelum fold berikutnya
        del model, trainer, train_ds, val_ds, test_ds
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()

    del tokenizer
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()


  SKEMA: binary  (2 kelas)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22831.92it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing 


--- binary | fold 0 | train=1440 val=160 test=400 ---


c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Non-negatif,F1 Negatif
1,0.712517,2.045057,0.975000,0.493671,0.987342,0.000000
2,1.052996,1.673510,0.975000,0.493671,0.987342,0.000000
3,0.636646,1.500468,0.975000,0.493671,0.987342,0.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.33it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.33it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.56it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


... (22 outputs left out)


--- 3class | fold 0 | train=1440 val=160 test=400 ---


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Negatif,F1 Netral,F1 Positif
1,1.024703,0.972346,0.737500,0.399957,0.000000,0.839216,0.360656
2,0.706955,1.017174,0.906250,0.517016,0.000000,0.951049,0.600000
3,0.534665,1.160502,0.906250,0.531955,0.000000,0.950704,0.645161


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.42it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.82it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.60it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

## Simpan & ringkas hasil

Metrik dirata-rata di 5 fold (mean ± std). Fokus utama: **f1_macro** dan **f1_negatif** (kelas minoritas), bukan accuracy.

In [ ]:
results_df = pd.DataFrame(all_fold_results)
results_df.to_csv(RESULTS_DIR / 'cv_results_per_fold.csv', index=False)

metric_cols = [c for c in results_df.columns if c.startswith(('accuracy', 'f1_', 'precision_', 'recall_'))]
summary = (
    results_df.groupby('scheme')[metric_cols]
    .agg(['mean', 'std'])
)
summary.to_csv(RESULTS_DIR / 'cv_results_summary.csv')

with open(RESULTS_DIR / 'cv_reports_detail.json', 'w', encoding='utf-8') as f:
    json.dump(
        {f'{s}_fold{i}': v for (s, i), v in all_fold_reports.items()},
        f, indent=2, ensure_ascii=False,
    )

print('Tersimpan di data/hasil_cv/:')
print(' - cv_results_per_fold.csv   (metrik tiap fold)')
print(' - cv_results_summary.csv    (mean ± std per skema)')
print(' - cv_reports_detail.json    (classification_report + confusion matrix lengkap)')
print()
for scheme in SCHEMES:
    sub = results_df[results_df['scheme'] == scheme]
    print(f'=== {scheme} (rata-rata {N_FOLDS} fold) ===')
    print(f'  accuracy : {sub["accuracy"].mean():.3f} ± {sub["accuracy"].std():.3f}')
    print(f'  f1_macro : {sub["f1_macro"].mean():.3f} ± {sub["f1_macro"].std():.3f}')
    for col in sub.columns:
        if col.startswith('f1_') and col != 'f1_macro':
            print(f'  {col:<16}: {sub[col].mean():.3f} ± {sub[col].std():.3f}')
    print()

## Catatan

- `SAVE_MODELS = False` secara default supaya tidak menghabiskan disk/waktu menyimpan 5 model per skema. Kalau ingin model akhir untuk dipakai di tahap ranking influencer (EDAS), latih ulang **satu model final** pakai seluruh data (tanpa k-fold) di notebook terpisah, atau set `SAVE_MODELS = True` dan pakai model dari fold dengan `f1_macro` tertinggi.
- Kalau muncul **CUDA out of memory**: turunkan `BATCH_TRAIN` ke 8, atau `max_len` di `config.json` (skema itu) kalau terlalu panjang.
- Bandingkan hasil `cv_results_summary.csv` ini dengan S0 (baseline tanpa fine-tune) dan S1 (preprocessing fix) yang sudah dihitung sebelumnya, pakai fold/split yang sama supaya adil.